# GuppyLM — Chat with a Fish

Download the pre-trained GuppyLM model from HuggingFace and chat with it.
No training needed — just run all cells.

**Model:** [arman-bd/guppylm-9M](https://huggingface.co/arman-bd/guppylm-9M) (8.7M params)

## 1. Setup

In [ ]:
!pip install -q torch tokenizers huggingface_hub

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
import os, shutil
if os.path.exists('/content/guppy'):
    shutil.rmtree('/content/guppy')
os.makedirs('/content/guppy')
os.chdir('/content/guppy')
print(f'Working dir: {os.getcwd()}')

## 2. Download Model

Download the pre-trained model, tokenizer, and inference code from HuggingFace.

In [ ]:
from huggingface_hub import hf_hub_download
import os

HF_REPO = 'arman-bd/guppylm-9M'

files = [
    ('checkpoints/best_model.pt', 'checkpoints'),
    ('checkpoints/config.json', 'checkpoints'),
    ('data/tokenizer.json', 'data'),
    ('model.py', '.'),
    ('config.py', '.'),
    ('inference.py', '.'),
]

for filename, subdir in files:
    os.makedirs(subdir, exist_ok=True)
    hf_hub_download(
        repo_id=HF_REPO,
        filename=filename,
        local_dir='.',
    )
    print(f'  Downloaded {filename}')

print(f'\nModel ready from {HF_REPO}')

## 3. Chat

Talk to Guppy. Each message is independent (single-turn).

In [ ]:
from inference import GuppyInference
import torch

engine = GuppyInference(
    'checkpoints/best_model.pt', 'data/tokenizer.json',
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

def chat(prompt):
    r = engine.chat_completion([{'role': 'user', 'content': prompt}], max_tokens=64)
    return r['choices'][0]['message'].get('content', '').strip()

In [ ]:
# Try different topics
tests = [
    'hi guppy',
    'are you hungry',
    'it is really hot today',
    'do you like bubbles',
    'what is the meaning of life',
    'tell me a joke',
    'the cat is looking at you',
    'do you love me',
    'what is the internet',
    'goodnight guppy',
]

for prompt in tests:
    reply = chat(prompt)
    print(f'You> {prompt}')
    print(f'Guppy> {reply}')
    print()

## 4. Interactive Chat

Type your own messages. Type `quit` to stop.

In [ ]:
print('Chat with Guppy (type quit to stop)')
print('=' * 40)
while True:
    try:
        prompt = input('You> ').strip()
    except (KeyboardInterrupt, EOFError):
        break
    if not prompt or prompt.lower() in ('quit', 'exit', 'q'):
        print('Guppy> bye. i will continue being a fish.')
        break
    reply = chat(prompt)
    print(f'Guppy> {reply}')
    print()